In [69]:

import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from uniformer import uniformer
from torchinfo import summary   
import torch
from torch import nn, einsum
from einops import rearrange
from einops.layers.torch import Reduce
from torchvision import transforms
import json
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from PIL import Image   
import os
from glob import glob
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [45]:
params={
    "image_size": 224,
    "frame_size": 25,
    "num_classes": 2,
    "dim": (64, 128, 256, 512),
    "depth": (3, 4, 8, 3),
    "batch_size": 4,
    "mhsa_types": ('l', 'l', 'g', 'g'),
    "epoch": 1000,
    "data_path": '../../data/',
    "second": '5sec',
    "class_name": '물과 비누로 손위생',
    "label_path": "../../data/label/check_list/",
    "image_channel": 3
}
params["second"]=f'{params["frame_size"]//5}sec'
params

{'image_size': 224,
 'frame_size': 25,
 'num_classes': 2,
 'dim': (64, 128, 256, 512),
 'depth': (3, 4, 8, 3),
 'batch_size': 4,
 'mhsa_types': ('l', 'l', 'g', 'g'),
 'epoch': 1000,
 'data_path': '../../data/',
 'second': '5sec',
 'class_name': '물과 비누로 손위생',
 'label_path': '../../data/label/check_list/',
 'image_channel': 3}

In [ ]:
file_list=[f"D{str(i+1).zfill(3)}" for i in range(200)]
remove_items = ['D151', 'D159', 'D187']
filtered_lst = [item for item in file_list if item not in remove_items]
trans = transforms.Compose([
    transforms.ToTensor(),
])

class CustomDataset(Dataset):
    """COCO Custom Dataset compatible with torch.utils.data.DataLoader."""

    def __init__(self, parmas, video, label):

        self.images = video
        self.args = parmas
        self.label = label

    def __getitem__(self, index):
        video1 = self.images[index,0]
        video2 = self.images[index,1]
        video3 = self.images[index,2]
        label = self.label[index]

        return video1,video2,video3, label

    def __len__(self):
        return len(self.images)



image_label = []
train_images = torch.zeros(len(filtered_lst),3,params['image_channel'],params['frame_size'],params['image_size'],params['image_size'])
for i in tqdm(range(len(filtered_lst)-1)):
    data_path=params['data_path']+filtered_lst[i]+'/*.png'
    with open(params['label_path']+filtered_lst[i+1]+'.json', 'r') as f:
        check = json.load(f)
    image_list_1 = glob(params['data_path']+params["second"]+'/'+params["class_name"]+'/'+filtered_lst[i]+'/1/*.png')
    image_list_1.sort()
    image_list_2 =[f.replace('/1/', '/2/') for f in image_list_1]
    image_list_3 =[f.replace('/1/', '/3/') for f in image_list_1]
    if check['행동'][params["class_name"]]==True:
        image_label.append(1)
    else:
        image_label.append(0)
    for j in range(params['frame_size']):
        train_images[i,0,:,j]=trans(Image.open(image_list_1[j]).convert('RGB').resize((params['image_size'], params['image_size'])))
        train_images[i,1,:,j]=trans(Image.open(image_list_2[j]).convert('RGB').resize((params['image_size'], params['image_size'])))
        train_images[i,2,:,j]=trans(Image.open(image_list_3[j]).convert('RGB').resize((params['image_size'], params['image_size'])))

train_dataset = CustomDataset(
    params, train_images, F.one_hot(torch.tensor(image_label)))
dataloader = DataLoader(
    train_dataset, batch_size=params['batch_size'], shuffle=True,drop_last=True)

  5%|▍         | 9/196 [00:04<01:25,  2.18it/s]


IndexError: list index out of range

In [78]:
params['data_path']+params["second"]+'/'+params["class_name"]+'/'+filtered_lst[i]+'/1/*.jpg'

'../../data/5sec/물과 비누로 손위생/D010/1/*.jpg'

In [3]:
model = uniformer.MultiVideoUniformer(
    num_classes = params['num_classes'],                 # number of output classes
    dims = params['dim'],         # feature dimensions per stage (4 stages)
    depths = params['depth'],              # depth at each stage
    mhsa_types = params['mhsa_types']   # aggregation type at each stage, 'l' stands for local, 'g' stands for global
).to(device)

video_size = (params['batch_size'], params['image_channel'], params['frame_size'], params['image_size'], params['image_size']) # (batch, channels, time, height, width)
optimizer = optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
summary(
    model,
    input_size=[
        (params['batch_size'], params['image_channel'], params['frame_size'], params['image_size'], params['image_size']),  # video1
        (params['batch_size'], params['image_channel'], params['frame_size'], params['image_size'], params['image_size']),  # video2
        (params['batch_size'], params['image_channel'], params['frame_size'], params['image_size'], params['image_size'])   # video3
    ],
    device=device
)

Layer (type:depth-idx)                                       Output Shape              Param #
MultiVideoUniformer                                          [4, 2]                    --
├─Uniformer: 1-1                                             [4, 128]                  --
│    └─Conv3d: 2-1                                           [4, 64, 13, 56, 56]       9,280
│    └─ModuleList: 2-2                                       --                        --
│    │    └─ModuleList: 3-1                                  --                        490,298
│    │    └─ModuleList: 3-2                                  --                        2,563,312
│    │    └─ModuleList: 3-3                                  --                        20,995,584
│    │    └─ModuleList: 3-4                                  --                        30,687,744
│    └─Sequential: 2-3                                       [4, 128]                  --
│    │    └─Reduce: 3-5                                      [4,

In [4]:
logits 

NameError: name 'logits' is not defined